In [1]:
## Load environment + imports
from dotenv import load_dotenv
import os

from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate


# ✅ REPLACE WITH THIS (free, local, no API key)
from langchain_ollama import ChatOllama
  # or "qwen3:8b" or "gemma3:4b"# ❌ REMOVE THIS (costs money, credits depleted)
# from langchain_google_genai import ChatGoogleGenerativeAI
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# ✅ REPLACE WITH THIS (free, local, no API key)
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2")   # or "qwen3:8b" or "gemma3:4b"
from langchain_core.output_parsers import StrOutputParser

load_dotenv(encoding="utf-16")

True

In [2]:
## Create LLM + Prompt + Chain
llm = ChatOllama(model="llama3.2") 
# llm = ChatAnthropic(model="claude-3-haiku-20240307")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are concise."),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

In [3]:
## STEP 5: TEST SYNC (invoke)
result = chain.invoke({
    "question": "What is RAG (Retrieval Augmented Generation) in 2 sentences?"
})

print(result)

RAG (Retrieval Augmented Generation) is a text generation model that uses retrieval-based search to augment the generated text with relevant information, improving its coherence and accuracy. This approach involves querying a retrieval module with the input prompt and retrieved passage, then using this information to refine the generated output.


In [4]:
## STEP 6: TEST STREAMING
for chunk in chain.stream({
    "question": "What is RAG (Retrieval Augmented Generation) in 2 sentences?"
}):
    print(chunk, end="", flush=True)

RAG (Retrieval Augmented Generation) is a natural language generation technique that uses retrieval-based models to generate text by first retrieving relevant information from a large corpus and then fine-tuning a generation model on the retrieved content. This approach leverages the strengths of both retrieval and generation models, allowing for more accurate and informative generated text.

In [9]:
## STEP 7: Pydantic Output Parser (Structured Output)
## Define schema
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


In [11]:
class Answer(BaseModel):
    summary: str = Field(description="Short explanation")
    confidence: float = Field(description="Confidence score between 0 and 1")

In [7]:
## Create Parser 
parser = PydanticOutputParser(pydantic_object=Answer)

In [8]:
## Update prompt to be more strict
prompt2 = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. "
        "You must return ONLY a raw JSON object that strictly matches the following schema: {format_instructions}. "
        "Do not include any introductory or concluding text, only the JSON."
    ),
    (
        "human",
        "{question}"
    )
]).partial(
    format_instructions=parser.get_format_instructions()
)

In [34]:
## Create structured chain
#  New / Correct Import
from langchain.output_parsers import OutputFixingParser

structured_llm = llm.with_structured_output(Answer)

# Use the fixing_parser in your chain instead of the standard parser
structured_chain = prompt2 | llm | fixing_parser

In [36]:
## Run structured output

result = structured_chain.invoke({
    "question": "What is RAG (Retrieval Augmented Generation) in 2 sentences?"
})

print(result)
print(result.summary)
print(result.confidence)

ValueError: Invalid input type <class 'dict'>. Must be a PromptValue, str, or list of BaseMessages.

In [58]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

# 1. Define your structure

class Answer(BaseModel):
    # Change the description to force a longer response
    summary: str = Field(description="A detailed 2-sentence explanation of the concept")
    confidence: float = Field(description="Confidence score between 0.0 and 1.0", ge=0.0, le=1.0)
# 2. Initialize the model
llm = ChatOllama(model="llama3.2")

# 3. Create a structured model instance
# This handles the schema enforcement natively, no parser needed
structured_llm = llm.with_structured_output(Answer)

# 4. Invoke directly
result = structured_llm.invoke("What is RAG (Retrieval Augmented Generation) in 2 sentences?")

print(result.summary)
print(result.confidence)

OutputParserException: Failed to parse Answer from completion {"summary": "RAG", "confidence": 8}. Got: 1 validation error for Answer
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=8, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

In [68]:
from langchain_core.prompts import ChatPromptTemplate

class Answer(BaseModel):
    # Change the description to force a longer response
    summary: str = Field(description="A detailed 2-sentence explanation of the concept")
    confidence: float = Field(description="Confidence score between 0.0 and 1.0", ge=0.0, le=1.0)
# 1. Update the prompt to be very specific about the scale
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. "
               "You must return ONLY valid JSON. "
              # "The 'confidence' field must be a float between 0.0 and 1.0. "
               #"Do not output values outside this range."
    ),
    ("human", "{question}")
])

# 2. Re-create the chain with the updated prompt
structured_llm = llm.with_structured_output(Answer, method="json_schema")
chain = prompt | structured_llm

# 3. Test again
result = chain.invoke("What is RAG (Retrieval Augmented Generation) in 2 sentences?")
print(f"Summary: {result.summary}")
print(f"Confidence: {result.confidence}")

Summary: RAG
Confidence: 0.8


In [69]:
result

Answer(summary='RAG', confidence=0.8)